In [ ]:
%cd latent-terrain-pytorch

In [ ]:
!pip install -r requirements.txt

In [ ]:
from __future__ import annotations

import argparse
import json
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import List, Dict, Any

import torch
import torchaudio

import numpy as np

from pipeline import audio as audio_utils
from pipeline import datasets as ds_utils
from pipeline import train as train_utils
from pipeline import reconstruct as recon_utils
from pipeline import eval as eval_utils
from pipeline import hpo
from scripts.factory import FlowDecWrapper, StableAudioOpenWrapper, Music2LatentWrapper, Codec
from scripts.run_pipeline import segments_to_latent_trajectories, pick_device, build_codec, CPPNHparams, RunConfig

from frechet_audio_distance import FrechetAudioDistance
frechet = FrechetAudioDistance(
    model_name="vggish",
    sample_rate=16000,
    use_pca=False, 
    use_activation=False,
    verbose=True
)

In [ ]:
config = RunConfig(
    dataset_dir="data/musdb_drums",
    dataset_name="drums",
    codec_name="sao",
    out_dir="outputs/drums",
    segment_seconds=20,
    target_sr=44100,
    stereo=False,
    save_models=False,
    save_audio=True,
    device="cuda",
    hpo_trials=30,
    hpo_epochs=400,
)
hparams = CPPNHparams(
    gauss_scale=float(16),
    epochs=int(2000),
    batch_size=int(64),
    c_max=int(512),
    mapping_size=int(256),
    lr=float(0.002),
)
fixed = {}
max_segments = 8

In [ ]:
device = pick_device(config.device)
print(device)
audio_dir = Path(config.dataset_dir)
if not audio_dir.exists():
    raise FileNotFoundError(f"Audio directory not found: {audio_dir}")

segment_samples = int(round(config.segment_seconds * config.target_sr))

# Build segments [B=1, C, T]
segments: List[torch.Tensor] = audio_utils.build_segment_dataset(audio_dir, target_sr=config.target_sr, segment_samples=segment_samples, use_stereo=config.stereo, max_segments=max_segments)
if len(segments) == 0:
    raise RuntimeError(f"No valid segments found in {audio_dir}")
print(f'segments: {len(segments)}')
# Codec
codec = build_codec(config.codec_name, device=device)

# Encode segments to trajectories [B, D, T_lat]
trajectories = segments_to_latent_trajectories(segments, codec, device=device)


In [ ]:
loader = ds_utils.build_dataloader_for_trajectory(trajectories[0], batch_size=hparams.batch_size, shuffle=False)
print(f'input trajectory: {next(iter(loader))[0].shape}, output trajectory {next(iter(loader))[1].shape}')

In [ ]:
best_params = hpo.tune_cppn_hparams_for_trajectory(trajectories[0], device=device, max_trials=50, epochs=1500, fixed=None)
print(f'best_params: {best_params["params"]}')

hparams.c_max = best_params['params']['c_max']
hparams.mapping_size = best_params['params']['mapping_size']
hparams.lr = best_params['params']['lr']
hparams.gauss_scale = best_params['params']['gauss_scale']

In [ ]:
c = 0
c_end = 1200
is_training = True
is_eval = True

# hparams.gauss_scale = 1

In [ ]:
models = []
psnrs: List[float] = []
mses: List[float] = []

data_codec = f'{config.dataset_name}_{config.codec_name}'
exp_name = f'{data_codec}_g{hparams.gauss_scale}_c{hparams.c_max}_m{hparams.mapping_size}'
exp_dir_ref = Path(config.out_dir) / f'{exp_name}_ref'
exp_dir_ref.mkdir(parents=True, exist_ok=True)
exp_dir_recon = Path(config.out_dir) / f'{exp_name}_recon'
exp_dir_recon.mkdir(parents=True, exist_ok=True)
exp_dir_model = Path(config.out_dir) / f'{exp_name}_model'
exp_dir_model.mkdir(parents=True, exist_ok=True)


for i, traj in enumerate(trajectories[c:c_end]):
    latent_dim = int(traj.shape[1]) if traj.dim() == 3 else int(traj.shape[-1])
    loader = ds_utils.build_dataloader_for_trajectory(traj, batch_size=hparams.batch_size, shuffle=False)
    if is_training:
        model = train_utils.train_fourier_cppn_for_trajectory(
            trajectory=traj,
            latent_dim=latent_dim,
            c_max=hparams.c_max,
            gauss_scale=hparams.gauss_scale,
            mapping_size=hparams.mapping_size,
            device=device,
            epochs=2000,
            lr=hparams.lr,
            loader=loader,
            print_every=max(1, hparams.epochs // 2),
        )
        models.append(model)
        torch.save(model.to('cpu'), str(exp_dir_model / f"{c+i}_model.pt"))
    if is_eval:
    # PSNR
        model = torch.load(str(exp_dir_model / f"{c+i}_model.pt")).to(device)

        psnr, mse, recon, ref = eval_utils.evaluate_model(model, 
                                          traj, 
                                          batch_size=128, 
                                          codec=codec, 
                                          reconstruct_audio=config.save_audio,
                                          device=device)
        torchaudio.save(str(exp_dir_ref / f"{c+i}_ref.wav"), ref[0].cpu(), config.target_sr) # type: ignore
        torchaudio.save(str(exp_dir_recon / f"{c+i}_recon.wav"), recon[0].cpu(), config.target_sr) # type: ignore
        psnrs.append(float(psnr))
        mses.append(float(mse))
        print(f"[Segment {c+i:04d}] PSNR={psnr:.2f} dB | MSE={mse:.6e}")

if is_eval:
    psnr_mean = float(np.mean(psnrs))
    psnr_p90 = float(np.percentile(psnrs, 90))
    print(f"PSNR (mean)={psnr_mean:.2f} dB | (p90)={psnr_p90:.2f} dB over {len(psnrs)} segments")

    fad_score_ref = frechet.score(
        f"/notebooks/latent-terrain-pytorch/{config.dataset_dir}", 
        f"/notebooks/latent-terrain-pytorch/{config.out_dir}/{exp_name}_ref", 
        dtype="float32"
    )
    fad_score_recon = frechet.score(
        f"/notebooks/latent-terrain-pytorch/{config.dataset_dir}", 
        f"/notebooks/latent-terrain-pytorch/{config.out_dir}/{exp_name}_recon", 
        dtype="float32"
    )

    summary = {
        "dataset": str(audio_dir),
        "codec": config.codec_name,
        "segments": len(segments),
        "psnr_mean": psnr_mean,
        "psnr_p90": psnr_p90,
        "ref_dir": str(exp_dir_ref),
        "recon_dir": str(exp_dir_recon),
        "ref_fad": fad_score_ref,
        "recon_fad": fad_score_recon,
        "device": device,
        "hparams": asdict(hparams),
        "config": asdict(config),
        "mse_per_segment": mses, 
    }

    with open(Path(config.out_dir) / f"{exp_name}_summary.json", "w") as f:
        json.dump(summary, f, indent=2)
